# 2026 CUMCM Problem C - Question 1

This notebook formulates and solves the day-ahead microgrid scheduling problem as a linear program (LP). It reads Attachment 1, validates and cleans the data, minimizes the daily grid-purchase cost, checks all constraints, summarizes the requested intervals, and writes the complete strategy to the official `result1.xlsx` template.

Required packages: `numpy`, `pandas`, `scipy`, `openpyxl`, and `matplotlib`.

## Linear programming model

For each 10-minute interval $t$, define:

- $G_t$: energy purchased from the external grid (kWh).
- $C_t$: energy sent from the microgrid bus to the battery (kWh).
- $D_t$: energy delivered from the battery to the microgrid bus (kWh).
- $W_t$: curtailed photovoltaic energy (kWh).
- $S_{t+1}$: stored energy at the end of the interval (kWh).

The objective is

$$\min \sum_t p_t G_t.$$

The interval energy balance is

$$G_t + PV_t + D_t = L_t + C_t + W_t.$$

The battery state transition is

$$S_{t+1}=S_t+\eta_c C_t-\frac{D_t}{\eta_d}.$$

The model also enforces the charge/discharge power limit, the safe state-of-charge range, photovoltaic curtailment bounds, nonnegativity, and $S_{144}=S_0=6000$ kWh. Charge and discharge are both continuous LP variables. With positive grid prices, efficiency below one, and an available photovoltaic-curtailment variable, same-period cycling cannot improve the objective; the diagnostic section verifies that the computed optimum has no simultaneous charging and discharging.

In [ ]:
from collections import Counter
from datetime import datetime, time
from pathlib import Path
import re
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
from scipy.optimize import linprog
from scipy.sparse import csr_matrix, lil_matrix

# Set any of these three paths explicitly if automatic discovery is not suitable.
USER_ARCHIVE_PATH = None
USER_SOURCE_XLSX = None
USER_TEMPLATE_XLSX = None

ROOT = Path.cwd()
DATA_DIR = ROOT / "q1_data"
OUTPUT_DIR = ROOT / "q1_outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INTERVAL_HOURS = 10 / 60
INITIAL_SOC_KWH = 6000.0
MIN_SOC_KWH = 1200.0
MAX_SOC_KWH = 10800.0
MAX_BATTERY_POWER_KW = 5000.0
CHARGE_EFFICIENCY = 0.90
DISCHARGE_EFFICIENCY = 0.90
SOLVER_TOLERANCE = 1e-7

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

## 1. Locate or extract the input files

The cell below can use separately extracted Excel files or extract the two Question 1 files from `CUMCM2026Problems.zip`. It copies only the required members and does not modify the original archive.

In [ ]:
def first_existing_path(candidates):
    for candidate in candidates:
        if candidate is not None:
            path = Path(candidate).expanduser().resolve()
            if path.is_file():
                return path
    return None


source_candidates = [
    USER_SOURCE_XLSX,
    ROOT / "q1_data" / "attachment1.xlsx",
    ROOT / "C题" / "附件" / "附件1.xlsx",
    ROOT / "附件1.xlsx",
]
template_candidates = [
    USER_TEMPLATE_XLSX,
    ROOT / "q1_data" / "result1_template.xlsx",
    ROOT / "C题" / "附件" / "附件5" / "result1.xlsx",
    ROOT / "result1.xlsx",
]
archive_candidates = [
    USER_ARCHIVE_PATH,
    ROOT / "CUMCM2026Problems.zip",
    ROOT / "01-CUMCM2026Problems.zip",
    ROOT / "project_sources" / "CUMCM2026Problems.zip",
    ROOT / "project_sources" / "01-CUMCM2026Problems.zip",
]
archive_candidates.extend(sorted(ROOT.glob("*CUMCM2026Problems*.zip")))
archive_candidates.extend(sorted((ROOT / "project_sources").glob("*CUMCM2026Problems*.zip")) if (ROOT / "project_sources").exists() else [])

SOURCE_XLSX = first_existing_path(source_candidates)
TEMPLATE_XLSX = first_existing_path(template_candidates)
ARCHIVE_PATH = first_existing_path(archive_candidates)

if SOURCE_XLSX is None or TEMPLATE_XLSX is None:
    if ARCHIVE_PATH is None:
        raise FileNotFoundError(
            "Could not locate Attachment 1 and the result1 template. "
            "Set USER_SOURCE_XLSX and USER_TEMPLATE_XLSX, or set USER_ARCHIVE_PATH."
        )

    with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
        member_names = archive.namelist()
        source_member = next(
            (name for name in member_names if name.endswith("C题/附件/附件1.xlsx")),
            None,
        )
        template_member = next(
            (name for name in member_names if name.endswith("C题/附件/附件5/result1.xlsx")),
            None,
        )
        if source_member is None or template_member is None:
            raise FileNotFoundError("The archive does not contain the expected Question C files.")

        if SOURCE_XLSX is None:
            SOURCE_XLSX = DATA_DIR / "attachment1.xlsx"
            SOURCE_XLSX.write_bytes(archive.read(source_member))
        if TEMPLATE_XLSX is None:
            TEMPLATE_XLSX = DATA_DIR / "result1_template.xlsx"
            TEMPLATE_XLSX.write_bytes(archive.read(template_member))

SOURCE_XLSX = Path(SOURCE_XLSX).resolve()
TEMPLATE_XLSX = Path(TEMPLATE_XLSX).resolve()

print(f"Source data: {SOURCE_XLSX.name}")
print(f"Result template: {TEMPLATE_XLSX.name}")

## 2. Data cleaning and validation

Attachment 1 contains a mixed-type time column: some cells are imported as `datetime.time` objects and others as strings. This step standardizes all time values, converts the three numerical columns explicitly, checks missing and duplicate values, and verifies the expected 144 consecutive 10-minute timestamps.

The source contains 144 rows labelled from `0:10` through next-day `0:00`, while the official template also contains 144 rows. The notebook preserves their official row order one-to-one and treats the rows as 144 consecutive decision periods. Four-hour totals use six consecutive blocks of 24 rows.

In [ ]:
SOURCE_COLUMNS = {
    "时间": "source_time",
    "电价": "price_yuan_per_kwh",
    "小区负载": "load_kw",
    "光伏发电预测功率": "pv_kw",
}


def normalize_time_label(value):
    if pd.isna(value):
        raise ValueError("A missing time value was found.")
    if isinstance(value, (datetime, pd.Timestamp)):
        return f"{value.hour:02d}:{value.minute:02d}"
    if isinstance(value, time):
        return f"{value.hour:02d}:{value.minute:02d}"

    text = str(value).strip().replace("：", ":")
    if text in {"0:00+1", "00:00+1"}:
        return "24:00"

    match = re.fullmatch(r"(\d{1,2}):(\d{2})(?::\d{2})?", text)
    if match is None:
        raise ValueError(f"Unsupported time value: {value!r}")
    hour, minute = map(int, match.groups())
    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError(f"Invalid time value: {value!r}")
    return f"{hour:02d}:{minute:02d}"


def time_label_to_minutes(label):
    if label == "24:00":
        return 24 * 60
    hour, minute = map(int, label.split(":"))
    return hour * 60 + minute


raw_data = pd.read_excel(SOURCE_XLSX)
raw_time_types = Counter(type(value).__name__ for value in raw_data.iloc[:, 0])

missing_headers = [column for column in SOURCE_COLUMNS if column not in raw_data.columns]
if missing_headers:
    raise ValueError(f"Missing source columns: {missing_headers}")

data = raw_data[list(SOURCE_COLUMNS)].rename(columns=SOURCE_COLUMNS).copy()
data["source_time"] = data["source_time"].map(normalize_time_label)
data["time_minute"] = data["source_time"].map(time_label_to_minutes)

numeric_columns = ["price_yuan_per_kwh", "load_kw", "pv_kw"]
for column in numeric_columns:
    data[column] = pd.to_numeric(data[column], errors="coerce")

if data[numeric_columns].isna().any().any():
    invalid_counts = data[numeric_columns].isna().sum()
    raise ValueError(f"Missing or nonnumeric entries were found:\n{invalid_counts}")
if (data[numeric_columns] < 0).any().any():
    raise ValueError("Price, load, and photovoltaic power must be nonnegative.")
if data["source_time"].duplicated().any():
    raise ValueError("Duplicate source timestamps were found.")
if len(data) != 144:
    raise ValueError(f"Expected 144 ten-minute rows, but found {len(data)}.")

expected_minutes = np.arange(10, 24 * 60 + 1, 10)
if not np.array_equal(data["time_minute"].to_numpy(), expected_minutes):
    raise ValueError("The source timestamps are not the expected consecutive ten-minute sequence.")

template_purchase = pd.read_excel(TEMPLATE_XLSX, sheet_name="计划购电量")
template_intervals = template_purchase.iloc[:, 0].astype(str).str.strip()
if len(template_intervals) != len(data) or template_intervals.duplicated().any():
    raise ValueError("The purchase sheet in the template does not contain 144 unique interval rows.")

data.insert(1, "result_interval", template_intervals.to_numpy())
data["load_kwh"] = data["load_kw"] * INTERVAL_HOURS
data["pv_kwh"] = data["pv_kw"] * INTERVAL_HOURS

cleaning_report = pd.Series({
    "Source rows": len(data),
    "Original time value types": dict(raw_time_types),
    "Missing numeric values": int(data[numeric_columns].isna().sum().sum()),
    "Duplicate timestamps": int(data["source_time"].duplicated().sum()),
    "First standardized time": data["source_time"].iloc[0],
    "Last standardized time": data["source_time"].iloc[-1],
}, name="Value")
display(cleaning_report.to_frame())
display(data.head())

## 3. Build and solve the LP

All power observations are converted to interval energy by multiplying by $1/6$ hour. The 5000 kW battery limit therefore becomes $5000/6$ kWh per interval.

In [ ]:
def solve_day_ahead_lp(clean_data):
    period_count = len(clean_data)
    max_interval_battery_energy = MAX_BATTERY_POWER_KW * INTERVAL_HOURS

    grid_slice = slice(0, period_count)
    charge_slice = slice(period_count, 2 * period_count)
    discharge_slice = slice(2 * period_count, 3 * period_count)
    curtailment_slice = slice(3 * period_count, 4 * period_count)
    soc_slice = slice(4 * period_count, 5 * period_count)
    variable_count = 5 * period_count

    price = clean_data["price_yuan_per_kwh"].to_numpy(dtype=float)
    load_energy = clean_data["load_kwh"].to_numpy(dtype=float)
    pv_energy = clean_data["pv_kwh"].to_numpy(dtype=float)

    objective = np.zeros(variable_count)
    objective[grid_slice] = price

    # There are T energy-balance equations, T state-transition equations,
    # and one terminal-state equation.
    equality_matrix = lil_matrix((2 * period_count + 1, variable_count))
    equality_rhs = np.zeros(2 * period_count + 1)

    for period in range(period_count):
        # Grid + PV + discharge = load + charge + curtailed PV.
        equality_matrix[period, period] = 1.0
        equality_matrix[period, period_count + period] = -1.0
        equality_matrix[period, 2 * period_count + period] = 1.0
        equality_matrix[period, 3 * period_count + period] = -1.0
        equality_rhs[period] = load_energy[period] - pv_energy[period]

        # End SOC = previous SOC + eta_c * charge - discharge / eta_d.
        state_row = period_count + period
        equality_matrix[state_row, 4 * period_count + period] = 1.0
        equality_matrix[state_row, period_count + period] = -CHARGE_EFFICIENCY
        equality_matrix[state_row, 2 * period_count + period] = 1.0 / DISCHARGE_EFFICIENCY
        if period == 0:
            equality_rhs[state_row] = INITIAL_SOC_KWH
        else:
            equality_matrix[state_row, 4 * period_count + period - 1] = -1.0

    equality_matrix[2 * period_count, 5 * period_count - 1] = 1.0
    equality_rhs[2 * period_count] = INITIAL_SOC_KWH

    variable_bounds = (
        [(0.0, None)] * period_count
        + [(0.0, max_interval_battery_energy)] * period_count
        + [(0.0, max_interval_battery_energy)] * period_count
        + [(0.0, float(pv_energy[period])) for period in range(period_count)]
        + [(MIN_SOC_KWH, MAX_SOC_KWH)] * period_count
    )

    optimization = linprog(
        c=objective,
        A_eq=csr_matrix(equality_matrix),
        b_eq=equality_rhs,
        bounds=variable_bounds,
        method="highs",
    )
    if not optimization.success:
        raise RuntimeError(f"LP solver failed: {optimization.message}")

    values = np.asarray(optimization.x, dtype=float)
    values[np.abs(values) < 1e-9] = 0.0
    variable_slices = {
        "grid_purchase_kwh": grid_slice,
        "charge_kwh": charge_slice,
        "discharge_kwh": discharge_slice,
        "curtailed_pv_kwh": curtailment_slice,
        "end_soc_kwh": soc_slice,
    }
    variables = {name: values[index_slice] for name, index_slice in variable_slices.items()}
    return optimization, variables


optimization, variables = solve_day_ahead_lp(data)
print(f"Solver status: {optimization.message}")
print(f"Minimum daily purchase cost: {optimization.fun:,.6f} yuan")

In [ ]:
solution = data.copy()
for column, values_array in variables.items():
    solution[column] = values_array
solution["interval_cost_yuan"] = (
    solution["price_yuan_per_kwh"] * solution["grid_purchase_kwh"]
)

daily_purchase_kwh = solution["grid_purchase_kwh"].sum()
daily_cost_yuan = solution["interval_cost_yuan"].sum()

daily_summary = pd.Series({
    "Daily load (kWh)": solution["load_kwh"].sum(),
    "Daily photovoltaic energy (kWh)": solution["pv_kwh"].sum(),
    "Daily grid purchase (kWh)": daily_purchase_kwh,
    "Daily purchase cost (yuan)": daily_cost_yuan,
    "Total charge at bus (kWh)": solution["charge_kwh"].sum(),
    "Total discharge at bus (kWh)": solution["discharge_kwh"].sum(),
    "Curtailed photovoltaic energy (kWh)": solution["curtailed_pv_kwh"].sum(),
}, name="Optimal value")
display(daily_summary.to_frame())
display(solution.head())

## 4. Feasibility and consistency checks

In [ ]:
energy_balance_residual = (
    solution["grid_purchase_kwh"]
    + solution["pv_kwh"]
    + solution["discharge_kwh"]
    - solution["load_kwh"]
    - solution["charge_kwh"]
    - solution["curtailed_pv_kwh"]
).to_numpy()

previous_soc = np.r_[INITIAL_SOC_KWH, solution["end_soc_kwh"].to_numpy()[:-1]]
expected_end_soc = (
    previous_soc
    + CHARGE_EFFICIENCY * solution["charge_kwh"].to_numpy()
    - solution["discharge_kwh"].to_numpy() / DISCHARGE_EFFICIENCY
)
soc_transition_residual = solution["end_soc_kwh"].to_numpy() - expected_end_soc
simultaneous_periods = int(
    ((solution["charge_kwh"] > 1e-6) & (solution["discharge_kwh"] > 1e-6)).sum()
)

validation = pd.Series({
    "Maximum absolute energy-balance error (kWh)": np.abs(energy_balance_residual).max(),
    "Maximum absolute SOC-transition error (kWh)": np.abs(soc_transition_residual).max(),
    "Minimum SOC (kWh)": solution["end_soc_kwh"].min(),
    "Maximum SOC (kWh)": solution["end_soc_kwh"].max(),
    "Terminal SOC (kWh)": solution["end_soc_kwh"].iloc[-1],
    "Maximum charging power (kW)": solution["charge_kwh"].max() / INTERVAL_HOURS,
    "Maximum discharging power (kW)": solution["discharge_kwh"].max() / INTERVAL_HOURS,
    "Simultaneous charge/discharge periods": simultaneous_periods,
}, name="Value")
display(validation.to_frame())

assert np.abs(energy_balance_residual).max() <= SOLVER_TOLERANCE
assert np.abs(soc_transition_residual).max() <= SOLVER_TOLERANCE
assert solution["end_soc_kwh"].between(
    MIN_SOC_KWH - SOLVER_TOLERANCE, MAX_SOC_KWH + SOLVER_TOLERANCE
).all()
assert abs(solution["end_soc_kwh"].iloc[-1] - INITIAL_SOC_KWH) <= SOLVER_TOLERANCE
assert solution["charge_kwh"].max() <= MAX_BATTERY_POWER_KW * INTERVAL_HOURS + SOLVER_TOLERANCE
assert solution["discharge_kwh"].max() <= MAX_BATTERY_POWER_KW * INTERVAL_HOURS + SOLVER_TOLERANCE
assert simultaneous_periods == 0
assert abs(daily_cost_yuan - optimization.fun) <= 1e-5
print("All LP feasibility and consistency checks passed.")

## 5. Results requested in Tables 1 and 2

In [ ]:
requested_intervals = [
    "10:00-10:10",
    "12:00-12:10",
    "14:00-14:10",
    "16:00-16:10",
    "18:00-18:10",
    "20:00-20:10",
]

table_1 = (
    solution.set_index("result_interval")
    .loc[requested_intervals, ["grid_purchase_kwh"]]
    .reset_index()
)
table_1.loc[len(table_1)] = ["Daily grid purchase", daily_purchase_kwh]
table_1.loc[len(table_1)] = ["Daily purchase cost (yuan)", daily_cost_yuan]

four_hour_labels = [
    "0:00-4:00",
    "4:00-8:00",
    "8:00-12:00",
    "12:00-16:00",
    "16:00-20:00",
    "20:00-24:00",
]
rows_per_block = int(round(4 / INTERVAL_HOURS))
block_rows = []
for block_index, block_label in enumerate(four_hour_labels):
    start_row = block_index * rows_per_block
    stop_row = (block_index + 1) * rows_per_block
    block = solution.iloc[start_row:stop_row]
    block_rows.append({
        "Time block": block_label,
        "Charge (kWh)": block["charge_kwh"].sum(),
        "Discharge (kWh)": block["discharge_kwh"].sum(),
    })
table_2 = pd.DataFrame(block_rows)

print("Table 1")
display(table_1)
print("Table 2")
display(table_2)
print(f"SOC at 0:00: {INITIAL_SOC_KWH:,.6f} kWh")
print(f"SOC at 24:00: {solution['end_soc_kwh'].iloc[-1]:,.6f} kWh")

## 6. Schedule visualization

In [ ]:
period_axis = np.arange(len(solution)) * INTERVAL_HOURS
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(period_axis, solution["load_kwh"], label="Load", linewidth=1.5)
axes[0].plot(period_axis, solution["pv_kwh"], label="Photovoltaic energy", linewidth=1.5)
axes[0].plot(period_axis, solution["grid_purchase_kwh"], label="Grid purchase", linewidth=1.2)
axes[0].set_ylabel("Energy per interval (kWh)")
axes[0].legend(ncol=3)
axes[0].grid(alpha=0.25)

axes[1].plot(period_axis, solution["charge_kwh"], label="Charge", color="tab:green")
axes[1].plot(period_axis, -solution["discharge_kwh"], label="Discharge", color="tab:red")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Battery energy (kWh)")
axes[1].legend(ncol=2)
axes[1].grid(alpha=0.25)

axes[2].plot(period_axis, solution["end_soc_kwh"], label="End-of-interval SOC", color="tab:purple")
axes[2].axhline(MIN_SOC_KWH, color="tab:red", linestyle="--", linewidth=1, label="SOC limits")
axes[2].axhline(MAX_SOC_KWH, color="tab:red", linestyle="--", linewidth=1)
axes[2].set_xlabel("Hour in the 24-hour horizon")
axes[2].set_ylabel("Stored energy (kWh)")
axes[2].legend(ncol=2)
axes[2].grid(alpha=0.25)

axes[2].set_xticks(np.arange(0, 25, 2))
fig.suptitle("Optimal Question 1 schedule")
fig.tight_layout()
plt.show()

## 7. Write the complete strategy to `result1.xlsx`

The official template is copied before any values are written, so the original template remains unchanged. Purchase values are written in the official 144-row order. The six four-hour charge/discharge totals and the two boundary SOC values are written to the second sheet.

In [ ]:
RESULT_XLSX = OUTPUT_DIR / "result1.xlsx"
shutil.copy2(TEMPLATE_XLSX, RESULT_XLSX)

workbook = load_workbook(RESULT_XLSX)
purchase_sheet = workbook["计划购电量"]
battery_sheet = workbook["充放电量"]

template_labels_in_workbook = [
    str(purchase_sheet.cell(row=row + 2, column=1).value).strip()
    for row in range(len(solution))
]
if template_labels_in_workbook != solution["result_interval"].tolist():
    raise ValueError("The template interval order changed before export.")

for row_index, purchase_value in enumerate(solution["grid_purchase_kwh"], start=2):
    purchase_sheet.cell(row=row_index, column=2).value = round(float(purchase_value), 6)

for row_index, block_result in enumerate(block_rows, start=2):
    expected_label = str(battery_sheet.cell(row=row_index, column=1).value).strip()
    if expected_label != block_result["Time block"]:
        raise ValueError(f"Unexpected battery block label at row {row_index}: {expected_label}")
    battery_sheet.cell(row=row_index, column=2).value = round(float(block_result["Charge (kWh)"]), 6)
    battery_sheet.cell(row=row_index, column=3).value = round(float(block_result["Discharge (kWh)"]), 6)

battery_sheet["E2"] = round(INITIAL_SOC_KWH, 6)
battery_sheet["E3"] = round(float(solution["end_soc_kwh"].iloc[-1]), 6)
workbook.save(RESULT_XLSX)

# Read the exported values back to verify that all required cells were populated.
check_workbook = load_workbook(RESULT_XLSX, data_only=True, read_only=True)
check_purchase_sheet = check_workbook["计划购电量"]
check_battery_sheet = check_workbook["充放电量"]
exported_purchase = [check_purchase_sheet.cell(row=row, column=2).value for row in range(2, 146)]
assert len(exported_purchase) == 144 and all(value is not None for value in exported_purchase)
assert all(check_battery_sheet.cell(row=row, column=2).value is not None for row in range(2, 8))
assert all(check_battery_sheet.cell(row=row, column=3).value is not None for row in range(2, 8))
assert check_battery_sheet["E2"].value is not None and check_battery_sheet["E3"].value is not None
check_workbook.close()

print(f"Saved and verified: {RESULT_XLSX.resolve()}")